# Exploration qualité des données

## Objectif

Analyser la qualité des données produites par le pipeline ISBN CollectionLens.

Sources étudiées :

- Nudger
- Google Books
- BNF
- OpenLibrary

## Import

In [1]:
from pathlib import Path
import sys
import json

import pandas as pd

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT / "src"))

In [2]:
from collectionlens.pipelines.isbn_pipeline import fetch_many_isbns_metadata

from collectionlens.api.google_books_api import (
    search_book_by_isbn as google_books_isbn,
)

from collectionlens.api.bnf_api import (
    search_book_by_isbn as bnf_isbn,
)

from collectionlens.api.open_library_api import (
    search_book_by_isbn as openlibrary_isbn,
)

from collectionlens.api.nudger_dataset import (
    load_nudger_dataset,
    build_nudger_search_function,
)

## Chargement nudger

In [3]:
nudger_path = (
    PROJECT_ROOT
    / "data"
    / "external"
    / "nudger"
    / "open4goods-isbn-dataset.csv"
)

df_nudger_source = load_nudger_dataset(
    csv_path=nudger_path,
    isbn_column="isbn",
)

nudger_isbn = build_nudger_search_function(df_nudger_source)

## Définition des sources

In [4]:
sources = {
    "nudger": nudger_isbn,
    "google_books": google_books_isbn,
    "bnf": bnf_isbn,
    "openlibrary": openlibrary_isbn,
}

## Chargement du dataset Benchmark

In [5]:
benchmark_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "extended_benchmark"
    / "benchmark_first_volume_by_series.csv"
)

df_benchmark = pd.read_csv(
    benchmark_path,
    dtype={"EAN": str},
    low_memory=False,
)

df_benchmark.head()

,Titre de la série,Type,Collection,Catégory,Titre de l'album,EAN,Tome,Date de publication,Editeur,Auteurs,...,Version numérique,A vendre,Numérotation,Prix d'achat,Cote,Etat,Ma note de l'album,Mon avis de l'album,Ma note de la série,Mon avis de la série
0,#les memes,album simple,NaN,BD,Chronique des âges farouches,9791038202733,1.0,2022-02-02T00:00:00.000Z,Fluide Glacial,Sylvain Frécon,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,103ème Escadrille de Chasse,NaN,NaN,Mangas,NaN,9782888906063,NaN,2013-09-21T00:00:00.000Z,Paquet,Takizawa Seiho,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2001 Nights Stories,NaN,NaN,Mangas,2001 Nights Stories,9782344057001,1.0,2023-10-04T00:00:00.000Z,Glénat,Yukinobu Hoshino,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20th Century Boys,album simple N&B,NaN,Mangas,NaN,9782845380790,1.0,2003-12-22T00:00:00.000Z,Panini,Naoki Urasawa,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20th Century Boys - Spin off,NaN,NaN,Mangas,NaN,9782809425659,NaN,2012-08-22T00:00:00.000Z,Panini,Naoki Urasawa,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
pipeline_isbns = (
    df_benchmark["EAN"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

print(f"Nombre d'ISBN à traiter : {len(pipeline_isbns)}")

Nombre d'ISBN à traiter : 1036


## Lancement du pipeline grandeur nature

In [7]:
raw_output_dir = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "pipeline_quality_baseline"
)

pipeline_results = fetch_many_isbns_metadata(
    isbns=pipeline_isbns,
    sources=sources,
    raw_output_dir=raw_output_dir,
)

print(f"Nombre de résultats pipeline : {len(pipeline_results)}")

Nombre de résultats pipeline : 1036


## Export global des résultats

In [8]:
pipeline_results_dir = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "pipeline_quality_baseline"
)

pipeline_results_dir.mkdir(
    parents=True,
    exist_ok=True,
)

pipeline_results_path = (
    pipeline_results_dir
    / "isbn_pipeline_results.json"
)

with pipeline_results_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        pipeline_results,
        file,
        ensure_ascii=False,
        indent=2,
        default=str,
    )

pipeline_results_path

WindowsPath('c:/Users/yoann/Documents/mes projets/CollectionLens V2/data/processed/pipeline_quality_baseline/isbn_pipeline_results.json')

## Vérification des JSON générés

In [9]:
json_files = list(raw_output_dir.rglob("*.json"))

print(f"Nombre de fichiers JSON générés : {len(json_files)}")

json_files[:10]

Nombre de fichiers JSON générés : 4144


[WindowsPath('c:/Users/yoann/Documents/mes projets/CollectionLens V2/data/raw/pipeline_quality_baseline/bnf/2351420225.json'),
 WindowsPath('c:/Users/yoann/Documents/mes projets/CollectionLens V2/data/raw/pipeline_quality_baseline/bnf/2845800711.json'),
 WindowsPath('c:/Users/yoann/Documents/mes projets/CollectionLens V2/data/raw/pipeline_quality_baseline/bnf/2910635169.json'),
 WindowsPath('c:/Users/yoann/Documents/mes projets/CollectionLens V2/data/raw/pipeline_quality_baseline/bnf/9781421532271.json'),
 WindowsPath('c:/Users/yoann/Documents/mes projets/CollectionLens V2/data/raw/pipeline_quality_baseline/bnf/9782016284407.json'),
 WindowsPath('c:/Users/yoann/Documents/mes projets/CollectionLens V2/data/raw/pipeline_quality_baseline/bnf/9782203062382.json'),
 WindowsPath('c:/Users/yoann/Documents/mes projets/CollectionLens V2/data/raw/pipeline_quality_baseline/bnf/9782203066427.json'),
 WindowsPath('c:/Users/yoann/Documents/mes projets/CollectionLens V2/data/raw/pipeline_quality_base

In [10]:
for source_name in sources.keys():
    source_dir = raw_output_dir / source_name
    files_count = len(list(source_dir.glob("*.json")))

    print(f"{source_name}: {files_count} fichiers JSON")

nudger: 1036 fichiers JSON
google_books: 1036 fichiers JSON
bnf: 1036 fichiers JSON
openlibrary: 1036 fichiers JSON


## Résultat du test grandeur nature

Le pipeline ISBN a été exécuté sur l’échantillon étendu issu de la collection réelle.

Les résultats sont sauvegardés :

- sous forme centralisée dans `data/processed/pipeline_quality_baseline/` ;
- sous forme de JSON bruts par source dans `data/raw/pipeline_quality_baseline/`.

Ces données serviront de base au notebook d’exploration qualité des données.

In [17]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().parents[1]

RAW_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "pipeline_quality_baseline"
)

## Chargement Json

In [16]:
print(RAW_DIR)

c:\Users\yoann\Documents\mes projets\CollectionLens V2\data\raw\isbn_pipeline


In [18]:
def load_source_jsons(
    source_dir: Path,
) -> pd.DataFrame:

    rows = []

    for json_file in source_dir.glob("*.json"):

        with open(
            json_file,
            encoding="utf-8",
        ) as file:

            payload = json.load(file)

        rows.append(payload["result"])

    return pd.DataFrame(rows)

In [19]:
df_google = load_source_jsons(
    RAW_DIR / "google_books"
)

df_bnf = load_source_jsons(
    RAW_DIR / "bnf"
)

df_openlibrary = load_source_jsons(
    RAW_DIR / "openlibrary"
)

df_nudger = load_source_jsons(
    RAW_DIR / "nudger"
)

## Vérification des volumes

In [21]:
for name, df in {
    "google_books": df_google,
    "bnf": df_bnf,
    "openlibrary": df_openlibrary,
    "nudger": df_nudger,
}.items():

    print(
        f"{name}: {len(df)}"
    )

google_books: 1036
bnf: 1036
openlibrary: 1036
nudger: 1036


## Couverture réelle

In [22]:
def coverage_rate(df):

    return round(
        df["found"].mean() * 100,
        2,
    )

In [23]:
coverage_df = pd.DataFrame(
    [
        ["Google Books", coverage_rate(df_google)],
        ["BNF", coverage_rate(df_bnf)],
        ["OpenLibrary", coverage_rate(df_openlibrary)],
        ["Nudger", coverage_rate(df_nudger)],
    ],
    columns=[
        "source",
        "coverage_rate",
    ],
)

coverage_df.sort_values(
    "coverage_rate",
    ascending=False,
)

,source,coverage_rate
3,Nudger,96.14
0,Google Books,84.75
1,BNF,71.53
2,OpenLibrary,16.89


## Analyse de couverture

Cette mesure permet d'évaluer la capacité de chaque source à répondre à une recherche ISBN issue de la collection réelle.

La couverture correspond au pourcentage d'ISBN pour lesquels la source retourne une réponse valide.

## Analyse des erreurs

In [24]:
def error_summary(df):

    return (
        df["error"]
        .fillna("success")
        .value_counts()
        .reset_index()
    )

In [25]:
error_summary(df_google)

,error,count
0,success,878
1,no_result,144
2,quota_exceeded,14


In [26]:
error_summary(df_bnf)

,error,count
0,success,741
1,no_result,295


In [27]:
error_summary(df_openlibrary)

,error,count
0,no_result,831
1,success,175
2,timeout,30


In [28]:
error_summary(df_nudger)

,error,count
0,success,996
1,no_result,40


## Analyse champs

In [29]:
FIELDS_TO_ANALYZE = [
    "title",
    "authors",
    "publisher",
    "published_date",
    "description",
]

In [30]:
def compute_completeness(
    df,
    fields,
):
    rows = []

    found_df = df[df["found"]]

    for field in fields:

        completion_rate = (
            found_df[field]
            .notna()
            .mean()
            * 100
        )

        rows.append(
            {
                "field": field,
                "completion_rate": round(
                    completion_rate,
                    2,
                ),
            }
        )

    return pd.DataFrame(rows)

In [31]:
compute_completeness(
    df_google,
    FIELDS_TO_ANALYZE,
)

,field,completion_rate
0,title,100.00
1,authors,100.00
2,publisher,15.83
3,published_date,99.66
4,description,66.06


In [32]:
compute_completeness(
    df_bnf,
    FIELDS_TO_ANALYZE,
)

,field,completion_rate
0,title,100.00
1,authors,100.00
2,publisher,99.46
3,published_date,98.52
4,description,99.46


In [33]:
compute_completeness(
    df_openlibrary,
    FIELDS_TO_ANALYZE,
)

,field,completion_rate
0,title,100.00
1,authors,100.00
2,publisher,87.43
3,published_date,87.43
4,description,0.00


In [34]:
compute_completeness(
    df_nudger,
    [
        "title",
        "publisher",
        "page_count",
        "format",
        "min_price",
    ],
)

,field,completion_rate
0,title,100.00
1,publisher,92.27
2,page_count,91.57
3,format,85.54
4,min_price,82.63


## Conclusion générale

L'exploration qualité confirme les conclusions du benchmark étendu.

Nudger constitue la meilleure source de référence pour les univers BD, Manga et Comics grâce à sa couverture exceptionnelle et à la richesse de ses métadonnées métier.

Google Books apporte une forte valeur ajoutée pour les descriptions et les informations bibliographiques générales.

La BNF fournit des métadonnées particulièrement fiables concernant les auteurs, éditeurs et dates de publication.

OpenLibrary reste une source secondaire pouvant apporter des informations complémentaires telles que les couvertures ou certains sujets.

Les résultats obtenus valident la stratégie multi-sources retenue pour CollectionLens :

1. Nudger
2. Google Books
3. BNF
4. OpenLibrary

Cette analyse confirme également la pertinence du pipeline ISBN développé lors du POC.